# Week 4-4 — Boosting 비교와 focused test error analysis

이 notebook에서 데이터 준비, feature set B, 시간 분할, 기존 LinearRegression/RandomForest 학습은 이전 실습의 검증된 코드를 재사용한다.

직접 작성할 부분은 다음 네 가지다.

1. `GradientBoostingRegressor` 학습과 validation 예측
2. 공통 validation 비교표와 최종 ML 후보 선택
3. 선택 모델의 train+validation 재학습과 잠긴 test 1회 평가
4. test 기간의 focused error analysis

## 0. 재사용 코드 — 데이터 로드와 feature set B 생성

아래 부분은 Week 4-2와 Week 4-3에서 이미 검증했으므로 그대로 실행한다.

In [109]:
import os
import pandas as pd
import numpy as np

data_path = '../../../data/week4_korea_cli.csv'

print('current working directory:', os.getcwd())
print('data path:', data_path)

ds = pd.read_csv(data_path)
ds['observation_date'] = pd.to_datetime(ds['observation_date'])
ds = ds.sort_values('observation_date').copy()

print('chronological order:', ds['observation_date'].is_monotonic_increasing)
ds.head()

current working directory: /Users/proudchris/Desktop/ML_modeling/answers/code/week4
data path: ../../../data/week4_korea_cli.csv
chronological order: True


,observation_date,KORLOLITOAASTSAM
0,1990-01-01,100.461861
1,1990-02-01,100.462284
2,1990-03-01,100.536374
3,1990-04-01,100.613811
4,1990-05-01,100.646672


In [110]:
# 원래 월별 행을 유지한 상태에서 target, baseline, feature를 먼저 생성한다.
ds['target_next_month'] = ds['KORLOLITOAASTSAM'].shift(-1).copy()
ds['baseline'] = ds['KORLOLITOAASTSAM'].copy()
ds['cli_lag1'] = ds['KORLOLITOAASTSAM'].shift(1).copy()
ds['cli_rolling3'] = ds['KORLOLITOAASTSAM'].rolling(window=3).mean()
ds['cli_diff1'] = ds['KORLOLITOAASTSAM'].diff(periods=1)

set_B = ['cli_lag1', 'cli_rolling3', 'cli_diff1']

required_columns = [
    'observation_date',
    'KORLOLITOAASTSAM',
    'target_next_month',
    'baseline',
    *set_B,
]

modeling_data = ds.dropna(subset=required_columns).copy()
modeling_data[required_columns].head()

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3,cli_diff1
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840,0.074089
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490,0.077438
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952,0.032861
5,1990-06-01,100.610137,100.477904,100.610137,100.646672,100.623540,-0.036535
6,1990-07-01,100.477904,100.254061,100.477904,100.610137,100.578238,-0.132233


## 1. 재사용 코드 — 공통 시간 분할

`test` 객체는 여기서 분리만 한다. 최종 후보를 validation으로 확정하기 전에는 test target, 예측값, metric을 확인하지 않는다.

In [111]:
train = modeling_data[
    (modeling_data['observation_date'] >= '2000-01-01') &
    (modeling_data['observation_date'] <= '2014-12-01')
].copy()

validation = modeling_data[
    (modeling_data['observation_date'] >= '2015-01-01') &
    (modeling_data['observation_date'] <= '2019-12-01')
].copy()

test = modeling_data[
    (modeling_data['observation_date'] >= '2020-01-01') &
    (modeling_data['observation_date'] <= '2024-11-01')
].copy()

In [112]:
X_train_B = train[set_B].copy()
X_validation_B = validation[set_B].copy()
y_train = train['target_next_month'].copy()
y_validation = validation['target_next_month'].copy()
baseline_validation_pred = validation['baseline'].copy()

print('X_train_B:', X_train_B.shape)
print('X_validation_B:', X_validation_B.shape)
print('features:', X_train_B.columns.tolist())

X_train_B: (180, 3)
X_validation_B: (60, 3)
features: ['cli_lag1', 'cli_rolling3', 'cli_diff1']


## 2. 재사용 코드 — 기존 두 ML 모델

Week 4-3과 동일한 feature set, train 행, RandomForest 설정을 그대로 사용한다.

In [113]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

model_linear = LinearRegression()
model_linear.fit(X_train_B, y_train)
linear_validation_pred = model_linear.predict(X_validation_B)

model_random_forest = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=5,
    random_state=42,
)
model_random_forest.fit(X_train_B, y_train)
random_forest_validation_pred = model_random_forest.predict(X_validation_B)

print(type(linear_validation_pred), linear_validation_pred.shape)
print(type(random_forest_validation_pred), random_forest_validation_pred.shape)

<class 'numpy.ndarray'> (60,)
<class 'numpy.ndarray'> (60,)


## C1 — GradientBoostingRegressor 학습과 validation 예측

다음 고정 설정을 사용한다.

- `n_estimators=100`
- `learning_rate=0.05`
- `max_depth=2`
- `min_samples_leaf=5`
- `random_state=42`

### 필수 출력

- 학습된 boosting 모델의 `type`
- validation 예측값의 `type`과 `shape`

In [114]:
from sklearn.ensemble import GradientBoostingRegressor

# TODO C1-1: 위 고정 설정으로 GradientBoostingRegressor 객체를 생성하세요.
model_boosting = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

# TODO C1-2: X_train_B와 y_train으로 학습하세요.
model_boosting.fit(X_train_B, y_train)

# TODO C1-3: X_validation_B의 예측값을 boosting_validation_pred에 저장하세요.
boosting_validation_pred = model_boosting.predict(X_validation_B)

# TODO C1-4: 모델 type, 예측값 type과 shape를 출력하세요.
type(model_boosting), type(boosting_validation_pred), boosting_validation_pred.shape

(sklearn.ensemble._gb.GradientBoostingRegressor, numpy.ndarray, (60,))

## C2 — 공통 validation 비교표

동일한 validation 60행에서 다음 네 후보를 비교한다.

- `persistence_baseline`
- `linear_regression_B`
- `random_forest_B`
- `gradient_boosting_B`

결과표 필수 컬럼: `model`, `evaluation_period`, `evaluation_rows`, `MAE`, `RMSE`

MAE 오름차순으로 정렬한다. 이 단계에서는 test metric을 계산하지 않는다.

In [115]:
# TODO C2-1: y_validation과 네 예측값을 사용해 공통 validation 비교표를 만드세요.
# 힌트: MAE는 절대오차의 평균, RMSE는 제곱오차 평균의 제곱근입니다.

validation_result = pd.DataFrame(index=['persistence_baseline', 'linear_regression_B', 'random_forest_B', 'gradient_boosting_B'])

validation_result.loc['persistence_baseline', 'model'] = 'baseline'
validation_result.loc['persistence_baseline', 'evaluation_period'] = validation.observation_date.max() - validation.observation_date.min()
validation_result.loc['persistence_baseline', 'evaluation_rows'] = len(validation)
validation_result.loc['persistence_baseline', 'MAE'] = (y_validation - baseline_validation_pred).abs().mean()
validation_result.loc['persistence_baseline', 'RMSE'] = np.sqrt(((y_validation - baseline_validation_pred)**2).mean())

validation_result.loc['linear_regression_B', 'model'] = 'linear'
validation_result.loc['linear_regression_B', 'evaluation_period'] = validation.observation_date.max() - validation.observation_date.min()
validation_result.loc['linear_regression_B', 'evaluation_rows'] = len(validation)
validation_result.loc['linear_regression_B', 'MAE'] = (y_validation - linear_validation_pred).abs().mean()
validation_result.loc['linear_regression_B', 'RMSE'] = np.sqrt(((y_validation - linear_validation_pred)**2).mean())

validation_result.loc['random_forest_B', 'model'] = 'RF'
validation_result.loc['random_forest_B', 'evaluation_period'] = validation.observation_date.max() - validation.observation_date.min()
validation_result.loc['random_forest_B', 'evaluation_rows'] = len(validation)
validation_result.loc['random_forest_B', 'MAE'] = (y_validation - random_forest_validation_pred).abs().mean()
validation_result.loc['random_forest_B', 'RMSE'] = np.sqrt(((y_validation - random_forest_validation_pred)**2).mean())

validation_result.loc['gradient_boosting_B', 'model'] = 'GB'
validation_result.loc['gradient_boosting_B', 'evaluation_period'] = validation.observation_date.max() - validation.observation_date.min()
validation_result.loc['gradient_boosting_B', 'evaluation_rows'] = len(validation)
validation_result.loc['gradient_boosting_B', 'MAE'] = (y_validation - boosting_validation_pred).abs().mean()
validation_result.loc['gradient_boosting_B', 'RMSE'] = np.sqrt(((y_validation - boosting_validation_pred)**2).mean())


# TODO C2-2: MAE 오름차순으로 정렬한 결과를 출력하세요.

validation_result = validation_result.sort_values(by='MAE', ascending=True)
validation_result

,model,evaluation_period,evaluation_rows,MAE,RMSE
linear_regression_B,linear,1795 days,60.0,0.010768,0.013730
persistence_baseline,baseline,1795 days,60.0,0.079752,0.093522
gradient_boosting_B,GB,1795 days,60.0,0.141976,0.181227
random_forest_B,RF,1795 days,60.0,0.194660,0.251603


## C3 — 최종 ML 후보 선택

Validation MAE가 가장 낮은 **ML 모델**을 선택한다. 동률이면 validation RMSE가 낮은 모델을 선택한다.

`persistence_baseline`은 비교 기준이므로 재학습할 ML 후보 선택에서는 제외한다.

### 필수 출력

- 선택한 최종 ML 모델명
- 선택 근거가 된 validation MAE와 RMSE

In [116]:
# TODO C3-1: validation_result만 사용해 최종 ML 후보를 선택하세요.
final_model_name = validation_result.iloc[0]

# TODO C3-2: 최종 모델명과 선택 근거 metric을 출력하세요.
# 주의: 아직 test metric을 확인하지 마세요.

final_model_name

model                            linear
evaluation_period    1795 days 00:00:00
evaluation_rows                    60.0
MAE                            0.010768
RMSE                            0.01373
Name: linear_regression_B, dtype: object

## C4 — Train+validation 재학습과 잠긴 test 1회 평가

선택이 끝난 후 train과 validation을 합친 240행으로 최종 후보와 동일한 종류·설정을 새로 학습한다.

그 다음에만 test를 열어 persistence baseline과 최종 ML 모델을 공통 test 행에서 한 번 비교한다.

결과표 필수 컬럼: `model`, `evaluation_period`, `evaluation_rows`, `MAE`, `RMSE`

In [117]:
train_validation = pd.concat([train, validation], axis=0).copy()
X_train_validation_B = train_validation[set_B].copy()
y_train_validation = train_validation['target_next_month'].copy()

print('refit rows:', len(train_validation))
print(
    'refit dates:',
    train_validation['observation_date'].min(),
    'to',
    train_validation['observation_date'].max(),
)

# TODO C4-1: final_model_name에 해당하는 모델을 같은 설정으로 새로 생성하세요.
final_model = ''
if final_model_name.name == 'linear_regression_B':
    final_model = LinearRegression()

# TODO C4-2: train+validation으로 final_model을 재학습하세요.
final_model.fit(X_train_validation_B, y_train_validation)

# 아래 두 변수는 최종 후보 확정 및 재학습이 끝난 뒤에만 사용하세요.
X_test_B = test[set_B].copy()
y_test = test['target_next_month'].copy()

# TODO C4-3: 최종 모델의 test 예측을 final_test_pred에 저장하세요.
final_test_pred = final_model.predict(X_test_B)


# TODO C4-4: persistence baseline과 최종 모델의 공통 test MAE/RMSE 표를 만드세요.
test_result =  pd.DataFrame(index=['persistence_baseline', 'linear_regression'])
test_result.loc['persistence_baseline', 'MAE'] = (y_test - test.baseline).abs().mean()
test_result.loc['persistence_baseline', 'RMSE'] = np.sqrt(((y_test - test.baseline)**2).mean())

test_result.loc['linear_regression', 'MAE'] = (y_test - final_test_pred).abs().mean()
test_result.loc['linear_regression', 'RMSE'] = np.sqrt(((y_test - final_test_pred)**2).mean())

test_result

refit rows: 240
refit dates: 2000-01-01 00:00:00 to 2019-12-01 00:00:00


,MAE,RMSE
persistence_baseline,0.194231,0.224456
linear_regression,0.017234,0.021142


## C5 — Focused test-period error analysis

다음 정의를 사용한다.

- `signed_error = target_next_month - final_model_prediction`
- `absolute_error = abs(signed_error)`
- signed error가 양수면 과소예측, 음수면 과대예측

분석표 필수 컬럼:

- `observation_date`
- `target_next_month`
- `final_model_prediction`
- `signed_error`
- `absolute_error`

In [118]:
# C5-1: test의 날짜와 target을 복사해 분석표를 만든다.
error_analysis = test[
    ['observation_date', 'target_next_month']
].copy()

# C5-2: 최종 모델의 예측값과 두 종류의 오차를 계산한다.
# signed_error > 0: 실제값이 예측값보다 큼 → 과소예측
# signed_error < 0: 실제값이 예측값보다 작음 → 과대예측
error_analysis['final_model_prediction'] = final_test_pred
error_analysis['signed_error'] = (
    error_analysis['target_next_month']
    - error_analysis['final_model_prediction']
)
error_analysis['absolute_error'] = error_analysis['signed_error'].abs()

print('test error-analysis rows:', len(error_analysis))
display(error_analysis.head())

# C5-3: 절대오차가 가장 큰 5개 행을 찾는다.
top5_errors = error_analysis.nlargest(5, 'absolute_error').copy()
display(top5_errors)

# C5-4: signed error의 부호로 과소예측과 과대예측을 센다.
underprediction_count = (top5_errors['signed_error'] > 0).sum()
overprediction_count = (top5_errors['signed_error'] < 0).sum()

print('top 5 underpredictions:', underprediction_count)
print('top 5 overpredictions:', overprediction_count)

# C5-5: 최대 absolute error 행의 feature date와 target month를 확인한다.
largest_error_row = top5_errors.iloc[0]
largest_error_feature_date = largest_error_row['observation_date']
largest_error_target_month = (
    largest_error_feature_date + pd.DateOffset(months=1)
)

print('largest-error feature date:', largest_error_feature_date)
print('largest-error target month:', largest_error_target_month)
print('target:', largest_error_row['target_next_month'])
print('prediction:', largest_error_row['final_model_prediction'])
print('signed error:', largest_error_row['signed_error'])
print('absolute error:', largest_error_row['absolute_error'])

test error-analysis rows: 59


,observation_date,target_next_month,final_model_prediction,signed_error,absolute_error
360,2020-01-01,99.222766,99.216544,0.006222,0.006222
361,2020-02-01,99.255908,99.241231,0.014677,0.014677
362,2020-03-01,99.318796,99.287430,0.031367,0.031367
363,2020-04-01,99.440145,99.405922,0.034223,0.034223
364,2020-05-01,99.654878,99.605560,0.049318,0.049318


,observation_date,target_next_month,final_model_prediction,signed_error,absolute_error
364,2020-05-01,99.654878,99.605560,0.049318,0.049318
391,2022-08-01,98.962862,99.011917,-0.049055,0.049055
401,2023-06-01,98.226187,98.266799,-0.040612,0.040612
387,2022-04-01,100.106825,100.146812,-0.039988,0.039988
371,2020-12-01,102.133518,102.096935,0.036583,0.036583


top 5 underpredictions: 2
top 5 overpredictions: 3
largest-error feature date: 2020-05-01 00:00:00
largest-error target month: 2020-06-01 00:00:00
target: 99.6548784496563
prediction: 99.60556015154464
signed error: 0.049318298111657555
absolute error: 0.049318298111657555


## 제출 전 자체 점검

- [ ] Boosting 외의 기존 전처리와 기존 모델 코드를 불필요하게 다시 작성하지 않았다.
- [ ] 네 후보를 같은 validation 60행에서 비교했다.
- [ ] 최종 후보 선택 전에 test metric을 확인하지 않았다.
- [ ] Persistence baseline을 재학습할 ML 후보 선택에서 제외했다.
- [ ] 선택 모델을 train+validation 240행으로 새로 학습했다.
- [ ] Test 결과를 본 뒤 모델 종류나 설정을 바꾸지 않았다.
- [ ] `signed_error = target - prediction`의 부호 해석이 일치한다.
- [ ] 오차 상위 5개 시점을 출력했다.
- [ ] Target 또는 미래 정보가 `X`에 포함되지 않았다.